# Расчет пролонгаций 

Notebook считает коэффициенты пролонгации за 2023 год по ТЗ для Junior Data Analyst. Его можно отправлять вместе с `_prolongation_report.xlsx`; если Excel/CSV рядом нет, notebook использует встроенный сжатый снимок исходных CSV и запускается автономно.


## Логика расчета

- AM берем из `prolongations.csv`, потому что по ТЗ этот источник первичен.
- Финансовые строки с одинаковым `id` суммируются по месяцам.
- `стоп`, `в ноль`, `end` и пустые значения считаются нулевой отгрузкой.
- Годовой коэффициент считается как отношение сумм за год, а не среднее месячных процентов.
- Точные дубли `id+month+AM` удаляются, конфликтующие `id+month` с разными AM не исправляются без бизнес-правила.


In [ ]:
from pathlib import Path
from io import StringIO
import base64
import gzip

import numpy as np
import pandas as pd
from IPython.display import display

TASK_DIR = Path.cwd()
DATA_DIR = TASK_DIR / 'ТЗ Аналитик (с 16.04.2026)'
REPORT_PATH = TASK_DIR / '_prolongation_report.xlsx'

# Embedded fallback: lets the notebook run in Colab/Jupyter even when only this .ipynb was uploaded.
# The same raw data is also included in _prolongation_report.xlsx on sheets
# "Исходные prolongations" and "Исходные financial".
EMBEDDED_PROLONGATIONS_CSV_GZ_B64 = """H4sIAHto92kC/82dS44cxxFA9zrFHICA8x+RSx/At9DCXtje+ACjkT8bwTQEGTAM2oYAH2BEciBKIqkrVN/ImdkkxZ7OElCRb+EFgeFM5+vIyPjlr+o3nz757e9/94dfP/nlrz5J4cn2Znt7erp9c7o9fXETXGi/+XK7P322vdp+OH2xPWzPb7a/tT/etR9ftx+3H9oP37e/37eWL0+329vt+fbq9JdPUo4cLCedwZ61P/6pffi71vDtgL1r97B9d7N9vT20H1+0/zy8w6jMKF+3tt+3f4Ox3Z8pTcom1pD19PSmif2qCXbX2p6Fah9o2qokrjjHKaxkAbVfEyhZyaBkWmawf3TF99+fB+BZY9+eFX9pE0PtkTUKXwlLzTI1+P+0xj++l+jvTcD7rqPeuf7lN9u/ukRD6vsPHcxSwdETj9p8BINESZmULZZKBjBBzKIG0hMLCAtg9IrOU9YvLhGK9820XnZpWxtrpPFlnRG7My9D/JxhNG0JLC6yuMTiMosrJC71MmmGM2ezGvaAr1ur258M7t9D1Bf9xy+HA75suPNn3sNylTnsqDOKU1Jr4iuJi4SHBl2HyF7AMWdrj7pm8ZnFCYpLaGfTTl8tOS165WCJjUCoKxUXmYiRkodAxROOKdz4eR/WJSrRcXHe5wSa556tHx65XCiQUKBEgTIDKjVyI1f2umcuRvZqLyvQ73X3UKqubh2SRAgIUDTkCMx0xAE1UFYg1JYADLGvhTa8ygLFZS5+l7Aj3T9Pn2/fto88nP58Difv40vL6zcXGh1ldXBkLRACWvSHFOjoxM6qa2HnOGBODqXQ00x2QufYJYSQ1mNIEDgbBoVK2qyRqh7YCVhAPaBoenJ62n75vA/TGRbtFqYcrG9uTWCHq8BWzUwwpgwgU12Z1l+DUqgSMKm8E9AYgqfkyq1wmo6hccEmsTjxKE6FxGXPdjZMccdqoL7tA7h2aNNzsGs9P2LW36M+50oxAErvI/fH/q2NfN8lWJIp9vA8w5mWM3IkZUt7XTVveraqYgo8NHPLZR3Sj0OAmirJcaPonedgpTp2DAtrY5ISieurg9xASAIsrQjaQQUtrR+QIGWrqFP1WSXW1cg6lWedKkhkgT4Dlpvg8D8qPBTI5mLRHa0d3kLuhcYMdLDU6Psmr0etcrc4B6wMaOzYX4JMM8B+sI6QR8IVyDQxosTpix2XoMOm0xci13s09u0uMYeW2FtWRjTSsx8DurY8U+ROgmCuB9o8sXRlZaRycivNS7r69oPr8EWJiCA+EuMSKLvNJUB2q/9voLFWzNjuWJNlvBsCZXWrEbifG1qInAqlWrlWiXlFpiqHqoSfqnMEZhzVu99+HIO5vNpBsmrmYNIvllCwIlPJTDVGP4LBoEqJoL5SpOSSvg12jTpc1PXYDWB6rr/GLMwRJ7RDp1PmPmPTtJJ9G9edMJ8hYaPAh3TmA4aqrPoLGbIiGBw0ZLKjsXhqCEbtSnVz1K8YLAkRvtRh+UIjmKyln6153TuytvrphMD0BfsLjGk6VfI6JNS0DlGfCK1ILZcYk9XkSMiSyyPMkZSpoQIdiYipieSFjvgVLaSw0njpm9NK4wWFjWuHqyM/zmVd+qQtcWnODChrpECPJLJM3F+d/rq9sW0r9wu59ubjmtdlc+uul2dAGgMDykt6HWvil83NBzkTpBqpkGq0V0aXIFM8lxUNj+kJopZrOYylkA9LJtNjNDPOQtnemCAxqBgZo8kTiQx7cTFCyi6MPB6yZs3Y4Ot1bjg++WEGXdNSqKie0snYRoDU27fAIDdNDpNKPDFiY0OgY5A15UvQsWNzcaW5Jn3c3Bz/UlmRZNwmfSyJ4XkBeiWFeRHc3pmxVr0uhV7r5JAUQQgpwvvSdXmRzikxxBoQ5cq1pVgzkke6VT3litUlCvWhjljVUnWeAl1Z9Tfto9+2Nr/YnjU1CjEaH/LgugrH5vV9+8+L0+f906u5sE9Jr3FW23UcbCyVcjCym+cdreshOG4Yo8y/RpmSzNwwTJmiZKqDet7/pcxVBcUViZjOXKR0Vvbsy6gzBW0/FMz2AzuW77bhH+OOzsFqf2QRpa0aQNVXP9XXkRquhoi6T4xc98bpvrXu6VzdJndOZVWacWvrs/bfN6e7j5/9Z723VcMezrT753alM2bFfvlxTz6jS4dCalD2NWjsclVySBTtbZGCCrePMypPhAbCFt1XqHeAhx9MUNBYMC7EzXHHTmZrv/HEyaUBxdV9nCnkVcEj1M9FAON0OzgaqYpL2c80gUPzMziT4Xg0V4479EwgUAloR0tF44pWNoT2jV3a5QS1PM2VxNWYUFzBg8G4/QyHLI8j+xIEWr3wnU5ouaZx16yPnaTAapYaIxb1Wpu3XYPAUKbmEhislh3JrLcKlMSNXSyqq9oPxsxgB+/yJXAstT9Mi9NX7U+OnMtmzDVFQVuLHoQl0gsSKJn0dXtyEGq/ec0ZiWbW5pKgOBHCS6vAnQSdfiyKgrL1ZVHS3vpTvIghiAHU2eRlPJbLYTXOMMc6lqaimB7T7WRdnOodJc+4VsGg+jwAQo0bXJBUQSiU+inKVPrWQqHGYc5ZBxcOdHK4vg4D4upcOtM6QsRsbPbSKvt2Y0ocbNx3o2CqBYSVwhoG6gU1J9QLJGJm67CAHeZSmTtZQS/wzmcsBSQuxwmQvsccF5LHo3ZaBXXKNohhbvcPrUS8/Zg27P2r0bu74QAPHz+1f1Xj2QOQcaiEsu8xw8NiafZcfeo8UC9PX4l2fNhKRTDTR6SvTX853Py9amsHEsHO5h3p/tt+9/KDt/+0Rnv25Edf9m7hMcxRppJrxzJsW8sBhAkIm76/LIKvHLMuSFeONX1hkXltG+xklQzCAgiT6csjlhJ1RUNaQkPa/GVrliMpGTSPcRtzPTn14yiYj9cEOpOqcJKpgCFDvXKS+f0AZFxByZ51Jrbc2MsDVhwbOQKZilMEA7gHc3H1QsJIb3AOTaEVTlQVrW7n75g7PqHfq9MO7jNHMEYWiaAnZdItKxofx8V4YAwDMtecvzBv4bg1O5tLDnTu5MEZhaBGkfeSpv1wlrLA8dxicp7OFh0C608E1t94ah5RfmPrGzGTzpXB+WJL6jvK+vji7/zdhkvX0zILrB4s6P4H0G+bhYOLAAA="""
EMBEDDED_FINANCIAL_CSV_GZ_B64 = """H4sIAHto92kC/91dzc4st3Hd6ykuZj1A+P+zzLPYWWSTbJK9bCPJwkZsGA4QJE6iAFknV7YEK7ZlA3qCuW8UVrFJHs50z7DZ/UnXESBg6nZ/3Ww2u1g/p0795fevt//48Ontqw//cPvq9vXt/bvbFx9+dPv89rsPP73e/u32hw8/vX3+4dMPP3mnhFLX2y9uX95+e3vf/dv/pL/71e19+Rd9vf1XOutX6arv02XKv/2Sz/jhIv3s9sd0/Mv++O1/l9///OEfb1/XIyS1836W7vXrDz/68IN6rf9M1/n6ww9xpOlf//3224d/u3sevfI8+uF5zMrzmOuff+97f/23f/U3nxh1vV60++YzpcRViMv17r/bz9OVfpDmNv0lXeXd7Wc0Denn79PP2+/o/un4+3TPL9Id/pDunN7FJ8766+2Pty/TP9FQfvouvZ/39Mzp/umkP6Y/fJ+EH1/p3+gfzrzvr+iS9Ljf3n1tNNfTn8XZ9G6k+OYzIU59N9rTOw/pusauXvc/07m/Tf//jv/mPV3s6zyN+YZpXtO9v6IVyhNNZ6UTPjEyXh/+S1/GVx/+jr6OfL18tS/48/nfd7T6089fJ+HLMpc+pPEF9c1nYeO5/4Nf5TK2f6IPgJ6a1EBWAf9GY+Pxv+er0tisT2Oju39NBz785LQXr8YW+iWmB9JCnfgi+d4ji/0i6d7m1EXkHC3ON1idSQ+9wXvywgy+J23TXLn1D2PvUua7Dr2haL7tO5pvPvPynJtKt/LZ/zNNNL9F/kx5/8yqo78AfZyaPs6LTTMvYx5SJxy5siwXXlZpJ6RZoGnwyz0/NoG25WWksyrCJxXx1e03t9+TkcH/f3X78t3tv2//mmYCrr+xd03fVm/f1sV0W/k2tzXbt6XZ9Na+yW3t9m1VWm5Cvc3Tuu3bkklXtNi5t9UqsOZPNzDpNVtaqbwP+NV9YNfX6pMmGdPSMpkvbtnRWPBy9e6zpgyPZEiP6qROlom+pG1WnjsOJ/XYjOCWee4Ld2mX+M6H4L/zIZjv/EVoyV+e/+azaJeVr9J3Z+Pqipsxkp3QJxjwxsnRr1ikr0fp8hmjdECL8P2Hvl2V5tKrMpdJkOEEHZbcq6QWDJgZx9+Mye/epfdtXuwmu1+XJVM+yjT35uwre/Zg01Q4b0++tCEnMSl+W3YBcrDkqpMzcXW7mI0CzcZ1T2f31V18Yh2F9B5EWPYTcoKNtqd93jxp0qZF5NItlKRVr+mG6UV5d9qmZXyOXNBiLY/SSSfdR0Y9qmicgXcpYxqLOv6p66R50wyqNIOhKDFFKjmcoMQ8fTu+jXrry9l3UdIinixhV76aNNyoD/tZVg8aCeziWC+av1OEo3cf0/gxr32+uwPb/IjGF+S9yrR5hXJlWgTGHn+uMLqTOtdWdCccvfvQrAbUlGFbU+65u1OalRVpdVuUiIm0dO3xNza6WnV6q7F8K6Qqo9tQYb9PN/u03f/f2bj7Nf38OSvhL5ICy+f8AUcxMr/PHYs3vDHashQwWk8SzI0hXWroFeRvVirPruY5wUC69+DEn31vxRmXSN5z/WCSFROKRR9YjaByLs5lSH8TS2wokHlZ1qVPX1wsV4sc3+oX6fQOayPZhiate1t93Ni+iHNmhEPJxoJbg0KO3MQIQbGPRzhgI0XeiskUK0/tTRPOiRApsvcUqVCVTCVv+fWl4ctkn3i2/owmB1Jenb8c0tXu/PRXMCu+6OREGJps59EIHJaOeRLDJipF7OoHsClkR0QpiJJ/PMLURsAzNJbFEi1efZESdJ5kH2bly5kakfOjuaIQmwHJildodHe0XbEV5jw3P5rWofCQtnLR1Y6GZFp8wyh11oiU2RGqba5X2DQOp0cxNi8O9lXy2GpCSIu2Fx8fkVdy2K6RMDEonDSKMQuHvipbX4/ZiOPP5dyTYzS2Riy5ZTXUsinoj1gwoq2peXuLZmxoPZPvWjzMC0V0ZN0ucCeL6aO3xX2y5G3XBKturzpfoCgy+kTt6lkocGLLrR2hvKIunjDZpiWqSleuuxqedWy+vqCZecf4ps/T0Tw9HmJPm8JJOZpAoZpIrmnQy7STkWxDzo6hcM4dKUg+GG2hxJyxbYl6c2oIjkcyvjEVq4G0nQnnZuw4d3HxCuwCCpfHVXU2a1DH7cgtrXQtiy0v4evoBdf07QlDMlb3eBnWSsoAoECXD1v69A7cSrBv9t6KAl8agqmMnrEo1MCoUetx0ekXroM8CDuxnPnQplmwZwRZKS09GLSj+SkvpxNoxfhiS6Jw3GIyYhwgVOI9XZIbf5MOX3unMyOTgrA65JCGuqduCavvaTInQmFGF2gNl/kmg9C8Ek74fqTnfAwZ5s2vGZUOJCjXHOu5tRRHbV0KaxUPjaNffsNdm9yCTRyO6Hn4zshBstZ+JyPhqO7qdjx/cz+aL1DolGm/IZw4qrGXQ99hEGCpBW2/k5GwmtPxzJt7PQgVzv9dJBnVSqrebrz9y4cfJQMkGR8f/j5HoEpIKmmfd50i4FC3VqNPfN5NLRmFmzg8ck+qURDAduiPbJ12RjSQzBZDurTERuipnVjs9e4IR3uLJW85OG9OGocLjKEWtOadW+7NsVq/bDcET0m3s2UOZAHcUS7ARHfvUsxOCON3L4riz9I29875vA46of7HaQGTHLDqWwQXj4BVXXwCVqWXUqEeNBemmNqcIQyQOWgWA7uYWfiLv/r+8e3Oj4a8aKEYZUrMi1DyaZKzViMnWMarC0saxxV9R4lT+ajv5sYaRuPOBp0TFhRY7FHBkRpIwCPneNB63Citrxeqli5oPJuWLKVzgrenDdQKt0eHt29FselmcvR1n5MiRpGofA9d83T0KdRoj3IaAgGKwkKuHrN+LerHr1yXiNauIRvGxQgFUSklSGUZRMlUZ5kRNMKsSfNOWNZpFPGqECT6Jh9/X6f3OZcUdXotP1j+6esPP07qqotDLb5/LTeQ9Gw12kp7nLJnbWs8mpVYWBcd2PrN0YsVs282wx6XSjBG+68gYaaUGn35y1UZtXTOVQmaOFjCJSADT4BJa9YExrG6ilmh5JSKVdJrzvz8wAetWXoL/rxs1CjcilS0rNPSwIwYOOgPELq5bIiULJLqnLlyw1BBxYV69X05AGf0ny8F2mTI2JC1gMR0ad3oZkiWD2kxqRczKJpwFWGJ9Ksos0DTaBfr7cRx+hDKRgHQt5UbzHtMKozGF3yASsQRhbc2zrmFRXDvHihFoA4Fpmqps7sQfM2VXY+E8KAZJ19GSMugmBpiVZVPv4WgQ7m0tav1nrMLyNlhY+riFNYG8AZg7oGjc0A0N6xF2cyMtWjyAoHjY2NQ0bwwJBjTGbRucciQfAhfHIrgdwq0ENMere1l2aNbwo4c77LJUbJCDhzBxF5AhwKFqakJ0oxb3OxCkR3EXiulgqxZXFiyf909bG32Y5NmPKjC8B0vljGRj63cImha0/fRxyNjWrP+eNlSUCepa+EXF5kKfSq2yHt5yhB8dLAhVD1H22RYEw4ag1afhw9zuugYtCozQPxeRe/yQ4J+6SLkhavBsAhrWnZ2XWi/uS7YLKwe2io0Z3Yp+GHVfuHSDA9Y2Gr/kVopVmtWUgaQDA/V6/uyIOPGcw6MuhaPnFwKo6WcWWtQ8Lv68rqV+XXC5EgIg5xvA57/JfhWjPFgHu28gV67AZfGxHNuYNZuIGMjNTl6A7t1g2jOeAIt5KstP0MrAV7UcumdkMZki4VJJQp1tFS8IDGv81CQOKV5eegbKqU3hCNgQ9coPSYVv60hKiI2kf7RBJuM7lEY8vk7oSJDW51DQUE9tIgMYrlrsIn0eY1akkVkinlEO44R543fDmw2adhpYfmor8rUaIWWV+uLFNLwLJYD7ufwoZFsWSM8bRSnDKVsM9h4NeTERkoTO3N16sjd5XB1DxUz5JeR7bEI70YKrr5ccPNPJSJTkjYsf9dLeOaUppB2GB8dIQreSYfqmeVpdlagbOSANUTenfDqKvkbkTnM4kqgTdksHDX6LFirlFmreQ7tDAZoF2udU0uuh/Htq740cpgNCOvpNNQKeXzFtGG7iq1UoOyD2RJIMek7P2j2bZox21bSJw4IcLElHdsWzDNDV9LObXEWY9HOhF+7ZxqaG8Gr6WC/vfoTF69bejK/5SrElnnpBTKJa8KQHNx7KOu+LIkYhaopMh+i9wUmDqVKzyXyKnR7266hKpJJHVBfke+u/Skrk59rLJ+oISuloWo3x4FrobIkZLwq8F4p2BvwLc1cs9EMuPQRXpA+5ZF88DvRI6A1UDhjHHv8JoGljhyIOGcgwYZ6hY0Y76GYXCDyiuUhNFXzVDSKpizAPRxlPrzqKmkiB6msAiVVcQ20I9xt5rvNoiB8uY9pNCIcWaxf6KZwTDNCLd9lLYa8T2vFl66ThyxbTpPUxJNoIAuuQZBYyVZpEcb+5jjwIa67Uh349qzfK0Xk86uWUAmvXQ1G7QiEkOwT7nzsY8Pd8kcooFLVfifQhiDQThBnvXg/ChJnxKqA0gnmp1wVVMcMZVqC5DFScWjYgylnCbpM2mZ1XhB6qsRaNcT8e/ayQW7sekbwGA9TLF6DZid2weORy+f0ApDrpP7QVJGC3gU5ynWobrm75PUsfbdx7L79+H7fB4Do5iL5r8Yu0SidtiCnL1MBM+PGQ8r8qcqmvTshqMVvnxVU0ibqDGFiBsZexUgGfO/vOQoZgrgyAUEFdOWySQdVHcqA8VyLdBmLWFwdrVrglXGxd2Unc2aJFmiUGGRhNXFFuFek+wIpypenr6WWZKfUcilyzUv6iV2HihqiIzWFxhVuJTPggUw1Q0A8mNoVvfnkiAGPbEyYmmsl5WuD4SIZLpfWOUfLOl1iGdYpW1XpE90+y0QitwPbeXh97EIj7ZEiRW87T9fbY9I0FmTPjpHzB8WG4M/RA9VLJXowj2iHuY9O7dtQ2HY4JULkgxtZgnGrLD8HM/ZLRx1u92RRXqRDE/sOXxq/nUEsZbf2lFuFCPHm9EAMG0arYd46jMLsQ29nlpE+Nnjs7jsWPlvQMsbOcpy/Oc6qtmctjCj8Snbg83T2b9Kf/dntl+l9+t2T/hghmArh0M666DhGdvicUGD+y2RLEPjpYhlwKq/S9DsJx0gUYazc2hZzgFh6uGbSUeoIIqK8i1ceLt6RkabB1pqH7ki456GZVQEy7li+Fy4kSU6IvZtW2cALzw3N6aRE2BUndVjkwrByh9NYQBeEPhFl6nP1BND+1EgRUT1UiBxzADW2wrkoJD3OHqWxTH3dEF6jYidHprW/Qrquxkwo5NNFw1vhOKbBLbDZHSkaT5/4q5Q9Zn4yH4poMMoSMDmvopyHNAprlewM1aqU8yWyEMjxtrEkXQalOiHzkzCKqmWj0wLWqXk0EuqUiApIBOBjjhJWViO2UffV5ZMwE4hZc0q/ukzkfspwNW4pUtZaXnUl0DsQpNwV6clFoEK24sHK+7UpHMmHiF12e64jV7HBTK1/JRzhbBtV+xdK64VQ4Lhbwtyh/ryjRAJhuO7vT+OJlGPKS3IqNYYvisDUpxLTr6GWVBFfgawSs8zU8iGtIZsljYCwEXOdV55UqTtfieuXK2Ef1zh392tjIf1SjYT7M0GaA//7l+B/STsVTd9C4zMnKUFE5V68lmRgTNiadHQNePPCkYwPGeJ5W5sIrDcKyktjAi1js/1qVFJbRkb7kkshVGXA9K93MceDtWY+MbWwqwaO9i90AZpWJuHUG/Zrd9I2yr0VxxdP23if6Zv17pRGr9FBzx39uMtOazpNFDwG2T3HhGfnSUpeG2dBst2xPlox7RQNb9kc63Xoo1WDjaycxnpiIW/mmFMMjii3ItxnSffB+3Zs6xwPWeFf3wetVnFvQAARcCgcCkPucWc3fSfOMtrNSPQkkm0kt75k+jzmCL5tYWf6ilvYmIwvrnSjr4XMHehrty8HLRwmhIV9EDPQ4qUwmSxWi46qcQTyHSrHCiFbKwyRAhH1CG0/7W8cGEl0mvegHtoFuqsRWLXMGOHsHOIEJyvxYs2WRahHYUyfXRM4GiIBweTjypGjNcIvP5aeu4SLIqpgoECdrMpKHEIeX30DxwUDkTveR0uQiE9zawIjkdWB5cdTs2Zr5YZ5sfH51RQk1VMYrDpSZal5sKN7HlkDjjhjJqtJraGWi56sBsZko6Hsg4MCvnOsKVVtAc9NRVQ7YjK42+z8HaCoO8O414RjVZZqZxrBPFrUb3/rjibSQlJxg0pyCo0wnWIK+yE1Dagc2/t+xKdN6NW4L8TLaVnq86vuMhOnDGaft5Eb1upCUUbhQ6ca8YE1ciFOIxCFBOK0OcPIzUORuC7OXrXikiN/da58uXkij7ChjDYt1eQd1g7HvWQ5bl6j473ElL7pTAYQvYXk2TlxC8bJc3F/O/ZMohylVixNNlsYpbfKCPy6UfG2457qDt7rjVtWJ0WMiWjMFs4ZuVT29DCTnJawJXemzILbIhOSCElmlFX3e+qz9A6JuQIyPCuIjDH3ezy+yQS/N2lLC7aV1OhGbMWCR+DI9JjCtUeRV8ZejVB5fV9jMM3DMxxSRuAt02EjZvitBNOo7TJ/pTpJmOTBGQ5XG0xbMLpOLNqEKfSXwLNjvkvVMnXUkjlDMEk3Ot2qbpUPLZwsnINCW1+4QMgWXWhNM32KB07nI8Jk9EHu4AKQFJyFNrlvITErsryqWt5k4l5pfiaG40/IV3HS7/mNPyoJ+i9C9x+zTRMwpf6jEntDZpQCTp9F3rrshnAQU6TETk8ktsjHRsRsvhvAeNNpg575nUQ7Sa0H5KLRdoyiUOFNJdLpbkBqa2qOwGG4QbYDaCgHDRwE1GqsJEKcQN/9RWuWYzE6j1RSAaDdXKnoINInIkQTWpgJkCS0+9oH9rm5L1qqDkgjGCACSr+r/5m05IZK4JegZ9dqqELSnwgOuUBQ6Nruar8hTNY9q2c5vbeoRdj8PfXWWwVmVuXiDuk8iTaO4CPX+vvsUzxqPwqwVe+mY+cPd4W8Up0QRPJDaHyyCmSlsSY4otBqXaI4QE07P5WYu9utShLpne8k7tkWzJpE1QBh4dee4wD0z4H/S5u0DpM6WVTcin05FSAWQ4psWpG0TyhF/mlmyjRwQ9d8Xm9MzNkSTnSr/QELfKCS2YJ/xv0TwtUUFH2LeHMnvuSHm0V72bQdGXtMqSJWi23ZlsSh6au7aGHwl3kXonC3DdfoD9tCgzm7Wn0tIBG8Uzi6yZnnWP5DCvjg2IKWQ5QilDGv9fgWAa8GbY5OIAPCrqI9dzL6PdEVm5Eo7KnXJVxaUuzAmHxQz4Auir6IOl2KrNrQkU5dNCdI08ZSkSr7gSpad/oytBwPZ1INMrrv5y7QzSKTycM3pY9QCHL5mMmh9XYhOGWcr9FZn/IhITwfWqIEavmzTaF7HXOdbbOmvWDT+lx84KBAp7K5bB4p1EPQ1aKdZzYW9eTecB6nZjB7QhwBsTsPQt07jgseKkx7ISBlEgg7mwIEsyegQcAGmTcgTrEWNJoEaljyr4iFy8R8ktXu6uOd+ahyKk0upA/BpvP9fXqVgmDpo9BxyckFf1V6ceAoIuf2P63a19SIyT8CpLuFXJIOTBZYMhB0RCnfKuhXT8sZp4qG2BSm6yj39U76U3q2qG3foETGOwN3HpHhQt5TNABwjgoZ1l+hcJ3AhKimG/mE6tOy69eCeGNlIcDBWCQoXJlzOKhsLpfKBEAG1SQCwXcq4JSAPbVXCh9Bq7BuBQ4Dug6HzNzXEY5U0jgW1OqlI5jNDCPW2JtSYg10cAfbHge7szUavRGDyY9mUDHf4npRzzTTlJrOOz/Lz09+uV7tbkHUWwaT9zV67315ga+17GQbRogDtPA8mn11cVuv4nByNu4FtC5k/JUIkPF5kI6YRmvHnXOSg6q+wix1z8A3vULD6qTXL1dg23HXNPdur4BIbAdYqIgUuYKyWM96wKvJdUEDdnFQsKEjt9/bqpEeZpMzl5GeLjs3kgOUrgLzbJrwaAqwTtmrDtmoE8GwFZiDixnoQv9Oi435djmOrplul/9YBMdW4P7R7yi75aCPL+PfFPacyh6SLLgNZqDUWZj6oqUY7fTHzQcL169jajgF+fn2ibPVYHJDHoqAEclJxU+GEkc1PpvxKiwbrV1axnGQI6a15U3hEFqaLfKqIxbQ6BbgMLXiCbYYAWk8uvwNMVfnC5APGc01xN0WPw1iXy7SRYjHeTQzeYTYh21O/0qxdy94SDzOYZci9bljbJuFUo6I0NjKQhjQO/BAVM9AhWLf1Y2Jr9Tod8YEhiY+tjLPpeyHtltt+m6GYDaSIBAcWdsWoDBtKctRcyf3XYpQDCSQL7/W7/GHaUCIERmn0MmKYVUwUCo3epoAAqZNYX6GxokZIBx+sGtRVL7zIAme1IiM2IPqpLYFy67z1WQ1wgj5NN+qkRtFcHA6gUosDDJ8bghI7/BEQEYnFI7V1PMjb5PqI8k3NB+4+x3hnNXf+KICVKwF8EjJVJBiRTj4gNFdz84aT6dvd9DWcNuCinecETjJF8WIZDoy9TlpPrYTx9mfOYZSkQmMZqgBFQfgTWRTzr0KzUkktT76F2ndrLJb8kjfOSevhQODC0/46s25Pfx83GOee3jWNxRCvCuc2Skcm43hOhYLvXaYIT6uLVwullKA1qkhMuboxcZVNa5PuAWFUUyHpcz1mXVHmOvvYVRHZyOEndUwEsMVmWOg9VXuQRvOd5WWfUPobUka4Ow59mA74yGAxmMWnQi90CsrDTdCrHuqhNogLrusKFDXrOR5YMHOWqWmPlRLH0qoQzOuQYjm8ANxNyXVY2eoI1DzqPdGQLn2Q2MTglZbo+7aEwRxzDDl0e2cHlp3cYWya5qby8VXlCK0oB3aix2XQzLYBcT/u7ZIBvE0XIajHiO8k6FDHvhzINYDXGmS65U6yF00jb+BLE+SAjjEp0gnzKx1aumPWnmsDgsOYO6HhROeMQSHKE4u2BLQsYUAIRwLI/uzzvSmcMaIHIwooD0eMTpAQIQGgSbEmsWy4daThrqzCY/KYXZXWaPtnL6aNb0264iRn0naDnTtnm8GoKH0gAuzgcCx5HDvhNCsvT5rNrVTCTGICmSfosLbLQBgzoB80jCe8lp7SKeicMLNlX/ZVJE/zmY0jktc31m5vrT8OKX5BRxjD0HWsA+HiG22UUC8+bzqkkLaCcuTB+kRr95268lcjLT77c3c9eGoKed7Lbl0X0NWldeCB9tuZ6qFONdG2aWwa869Vp/IbOjRBmrk0FZSMIt0r4ygAVLYmvxgwYBjVfvIG6Qx5gYrCvs/aNg1qi7n+hdsE1FpRjgC7Y+lH6I0u30NZugJrV9fDP5R6IyL6Y1Xmr0eLncVc745P65kL1E4b3RHgebnjMS73aqMweOlZ9F5o9jrGWronXLKKKToitzYNuJArYK+scaoq+UevMnu+TIN6lO8C9s7v+Av5YdsAFXW3xh3dLWC5CjFuyq8keJdAYWaUsPNf+S3UUhMDffoZnKfUuZnHCt0txB2ccggrLr2dLhfGkyhoaCgBd3I7wA61reOdQce3Mq4E6SHvSik7LIRz6T+79RGO7iZ0Y+9NopvNubIALPHuHBgLeeAUcso4vtCoScY2fqTAw8XQuw1Rk1FMhVoBLhobH2YMgflUTsxWtcbXliNsy3Y3jKcRWa2dm/MhRcAZWLQAjMNltXJrBWCsi+ESctV3LXOvqg7lPVekNUwy7XjhbXgoR4Fo9uK+IgESojYFgcJjVJhZqKGvnbiD7F6YQWiLjbUJbWiVyq0ixZUHRmfZkyLYdXdivtSSdWoSQT26jbRZWBWYB2wlABEbjrhmiNHlEPOMk8VWWRL15K4MC1FW8pDhG4xKx1chrvt1IovA8UL76/BovJjgrkjsTkkbBIAz0zE0xCJhASQ9LhOFTDDSIEkCaRvKwVpd5rDTR+TXTuHPYpSSDcBPGKu9qhFUZJYeEStwj5HYsYdbUBytbhp9hhjJmrClCINYXG0ZqPio3iGrA0KypJ5QdL7V3JZDJGIweKDEDJxq9XLEWXkNcrCR5u0jilBEqsj13TlTqmxnGZz5byVZTVVsC4VbshcspUv4NXV+5oTlm2h6qRdTTmNjBcKgU8XAXSQr8DR0gDtbqh7V4vU6QJyZf4mEYFsyEh5bO/3YnQv1FwvVTPwzHRfcvjOYeNHb5aGpYu7LoCO1ZE6B35ipOuVDjOs3Figfeq9RPuArhVOfRcT66BxkyagurLihFkahTrErjtCY21DB4KcjABOycvffUr1gKkn2rIz3VvaEsKCeH4UOI7nxNoRyo57jDYbrVekaWN5dzafN/7qxHKPjCqgSuecQVsu/5VG8EV1/FsBeXbe70a/FP+qBxoUg1urQ5hUOwJb2Wy6Oo5uYhQTbPtmL3H1pHALv+PHKs3CEc0wDJpbJrbuKJzOsfYNJVobjfR5+8wHiDAFgmspYifNz9F+YEomZDjC8+7HS/IYmVOpl2gT2RAMTM2U4IBufVBw0MSzE+ZnZbxXfKXHcFiSClsuAnVyQwAws2vwUFpw4Tn95FaCMGSZxYgdJRz8fX0LXBAnVoS5dgB2PDiqwK35uITr/MMPGdrMwt81ExSAzag5IAbkeQD2V9Y+7ZDcPgKyilNFNeQcoCkAMyNgJ4aKAiSsX63RprIphRGs1oV8blLcAreNOi58AJ0Qs3ldBRU/FmGWEiUuIKRQ6tp6gTn+wlkCcaMEdY4w24LWFGuKVFzJu2mOMC/c6JrD9qUdHpVzlXBUfwQFpodxi33BcPRomhkYTXwUpilh9qfOrjkQ1rgELOZn/NHSb28nknkO67ywxRLbzbWzBQG3/TL9OV7t5stIeaR7E36UwS41mxS8NwvxC3cQCYdepbIdUkNaeHJJ66tOCqO0Gjsw9z+rZpt1kFtioKjA9vFRqfkp8wvR0yZKbEoYK+V4Iqh7HMts7nXU04nmDiv4pyDMRynGq8czgwC4N7GDh0XYx3MtQo2/0k7jam8bCtvraloyH0MrVZJYxEohyBbSjR6Ibzmo2j6hCPW+90UMByZm0IZGjdu1PL6L6jRifjKuNFpNyAZfjWiE0fQCg3LUK4HDRyU4NgtGI8xCLhJpYNiuvU5PLKEUZpSVwc7TnILrCJi3JM5RVokbdNl1iZ7XyNcSiTTRy6zvJrxwdoRmMo2c4VISPGv1phJtArHRuBJyRuojexU/6GatK5rhGAPy/b+Ltd8biGPmKNSrhYDcletYhMubcRgZQ7Tqag3o9iLbH3+xtQi3EwZPs1i1MCbs5jQx4wA1TrEFZ65GLYk0naxna+9dd2Yb91e9MN25EGp8QEsJLr1c2v34TIfHqRKVWTScWXjHK4Vw1tyyJHT8kgXejcQQYwSmHKTzxi+c/KSXtaoSjbIdo2eJpW/NE4mXSrLgnguzzKziCe1p3leSdUp8IlwXSZQUtgjWQUyJEEpBlXCVijmjlqkXYn5hdDWb3BZZmRCJH8Uvq4JGotT0g0RjdofgJQZATqDZNOYs8qpJ8oSwt70P7bUWbb4jwuwSDH7ftI3gSzM5rHlRgzI7Yr+31S1FmCqAmYRGCeggUTl2ZLE2isU0Sa39uiJPwGCeCEBwOE36EUZZtpl/AEu3VUCmHBUUJPqYLbi6DYJpUo9095HhFe9A36bM5R3K+UXfWe2vbqUzmU37m58dlbQv+8xbxrXY9r6+W8Fb5DR5Kqg1Yd5Pt+Y508qrmrUL+h/9u9749y2zVOLz3PWLsG/5nBttDw6wLtjXplH2lmRXLNgKo2OHsvAKCVQocOpqfZ/sggK4WLJr1LlxTeIWge3MjsqAzJV2pnDoiXqglp/19O2rFhQtkOmOBhX8eLQFuSck9DrnkFgtWOwKoNE765rYSHXHXdGOhLsjEZtwtxXg7mrLGjwev7fZaXG1GLK1j+iWnECfP3SViLAc7F3SUKDgICwj8LkgtIALzcATYwckd3zFmZi3SXR86TNapRe5YFrujjvK4Ftoz9EJ8v5l4Ty22HLEjsk4FHE8lqZ2ZOkXHJitJWp+SYJR3MMjFX8NFE4dmRHw8zDQ3HmySfiOLL3KaEJbFrOKusUVavCPNUPHtlchmxhx7DP1AkIWCo1g6nZcsbadkFU+5HYDhm/8oVnZQXbMMVFdGk3ywx8SPOxWTAJh1wSNulQjIGtTmIxc6X2MX/9vZyLurYdOeo5zYz63eFIq5iCJq5JUcoEAc+FA+hhMNW0o9N8k4zz3MplubGfnXeeLyvguuUTZrA4cfbs/Ry3dkS86Q+XVkfHuDVB4oDjlOFPE8G0ruMKty6q7MKc/FqCIMuztmuzAovJ+S+A43+vzjgz7jLhKbvVGHdVjWHFeF9/7hLiKpLR8TzvABNARIYlmVZqmAoj7QQeKqTFLkoT7LjntSgmoaICQQ0XFcQJjwNUf8hrMYruXJmgsBBe5suFIopuZkTkDKgICdJsfxgu3S7m2RCpjMhR2aG0Sf7T1715JlWaXr1INEWkwZiZN10FCd+BCgWRvkip3C8X7bC6TWE57EjIDPR4zuKdRwRjs0s1tL4CszCPzcMVrPBGw72p3NQYNmxVhdwqEWD4zK7lAm1fWxDj3JY1r5IZMjC6BjbkVffk7Mj5vAXJSqbS7q5EjpdFb1HblyP6cls5+k1fAT0F1thoBhLW8m3MeeFptVN8J3AgdixXOE6zJPPLToBzn9VDYNRk1SIKjuEmwGZGUfi5V6JZSiOlQTNfV3EqQ5prYe/0kCEN2TXQ1xGQrXV+NMWOA4zttaevHG+rkolsPH11twESCDnBEazF0ZFAYuDSCsJDiZn5WxhHpFsqfGn0mDujb+80DKrijSULrkcx00hw45d+t5BUDoPVV63tpNgYUt6OsuU3WI4XS5K2IOH2Q+427zrfSwpOkbvdDae9+LtfZ45bofOvP3ksSWYG5RMih8RXMiMTXDBAjEvUiEijEZVfNPCiYjh0dwsc7p2e0+xMHNzxSoBMPky2xCbm0BOzJuybGMsxN83os3LxASjSFicSoJvS1WoDm0UIl4tsJuyeEQhIX1o+f8nO/S6uXjv/29tUSRlIYYOrCSAH7EH5EQl99iVGtycraKEZJRTCS0gnMt+KBjLDGZb2HYDCVZ7RGE9+SQN6CXhOOzNcejzsDCLEXjcGOYRZD5TVowa+8hvwtmMDBYSEcF0UdeyCiBBrtXW4gD7O4aAPSJEpKvOzeoDpzIUIKmNwuEXWGCXIlBwpa2yMD61uFOnzsRymuSnPlU36wpzJ/g0RXwYBGzzwk7ipD8VTFEs0lFLyV+ipjAVUkzW9MYXOwMlyDLgl15a5BFoCMTPuA9OVEovqTdtHWVsir1Et/bkegPldimKIANOmCVLPJQEzi5qVPInNr5LxQRRYDpndyffOsDXdkTreSZ9h/1viXvLfM1No6l2I3V3ZmjULbrh6TruNJYrrRehUCgFlUuLXKIUdnDbZkamU8DDxTKLkOFd8aXXBRbfd3EHGhlXa4G6wLTd/IdkEOkwmMmXm3JuQoIPSaqptRfwSEA6FOF9So/RcyvrlSAEazS1iQmAJCeGPCvA8T1Kg9yT0x0tedU13cWT152yGDQ5juUOXmlYosApW2uXyM3oJw+hpPcbqilXccFh18flSaL2TyO7pGMdYtgiVQ65npiHgtMDlSMRgoHFmJePsjHklRJLL5g0d1BNnlx4G5m7kjDQrOQ8w7dl1iMCTs8jqqjXmqQOutIj3JhawAdLLEGuXhgQfe0U6J0xcBmGdUCQ1HB8QZXFhWPLLYtaI1wIr8RLBY0bYlHH3qYS4hCfSmscVKMahW3zHGUBW6N52AvcEylMSv/o1qfD4HnjaIHXUCvBidAMiKj1Ax10ruLFBN0CPUrBNt4iKCQx6RPU8JuFrEHu1KTrrG/HzjbsyFSeVtHxzbfcvQczhVyz1sCrrD30lsNUqGZ/uw4MQ5/iPKDu1O8TI9bU9duyXJPqPf43cnSJPssa63vfKtBOr3P+Y8JGt7gmnwwOMdOrKDipqjvQKEuef2wXVWL/55euu/Sargz26//ESFV4HxTBJqwc3sai8iNqdrPcyIvtRgGETiCg+dB+8NrguvsfVjC+k6ZMlmBjpTOzRFyDIyFsYi/Ln1N1Ftx59WjTxjaxFhhVRlPXlOD+RGKkqPgzWIGcW/0Y/9EOc3smBrLyktXAZkMT9nDNkvJtB9Ye3nzqoBOqLCkOq76To+3P94UlU0z62vsQIHY5dHhem+Z7vVZ941Qwk8k2Vq1SJgOxzNCWyACDYkMQD2d25G/wfgPna1OyIBAA=="""

RU_MONTHS_LOWER = {
    1: 'январь', 2: 'февраль', 3: 'март', 4: 'апрель', 5: 'май', 6: 'июнь',
    7: 'июль', 8: 'август', 9: 'сентябрь', 10: 'октябрь', 11: 'ноябрь', 12: 'декабрь',
}
RU_MONTHS_TITLE = {
    1: 'Январь', 2: 'Февраль', 3: 'Март', 4: 'Апрель', 5: 'Май', 6: 'Июнь',
    7: 'Июль', 8: 'Август', 9: 'Сентябрь', 10: 'Октябрь', 11: 'Ноябрь', 12: 'Декабрь',
}
MONTH_NAME_TO_NUM = {v: k for k, v in RU_MONTHS_LOWER.items()}
REPORT_MONTHS = pd.date_range('2023-01-01', '2023-12-01', freq='MS')
REPORT_MONTH_LABELS = [f'{RU_MONTHS_TITLE[d.month]} {d.year}' for d in REPORT_MONTHS]


def month_label(dt: pd.Timestamp) -> str:
    return f'{RU_MONTHS_TITLE[dt.month]} {dt.year}'


def parse_period(value: str) -> pd.Timestamp:
    name, year = str(value).strip().lower().split()
    return pd.Timestamp(int(year), MONTH_NAME_TO_NUM[name], 1)


def parse_amount(value) -> float:
    if pd.isna(value):
        return 0.0
    text = str(value).strip().lower()
    if not text:
        return 0.0
    normalized = text.replace('\u00a0', '').replace(' ', '').replace(',', '.')
    if normalized in {'стоп', 'вноль', 'end', '-'}:
        return 0.0
    try:
        return float(normalized)
    except ValueError:
        return 0.0


def coeff(numerator: float, denominator: float):
    return np.nan if denominator == 0 else numerator / denominator


def read_embedded_csv(gz_b64: str) -> pd.DataFrame:
    raw = gzip.decompress(base64.b64decode(gz_b64)).decode('utf-8')
    return pd.read_csv(StringIO(raw))


def read_inputs():
    prolongation_candidates = [DATA_DIR / 'prolongations.csv', TASK_DIR / 'prolongations.csv']
    financial_candidates = [
        DATA_DIR / ' financial_data.csv', DATA_DIR / 'financial_data.csv',
        TASK_DIR / ' financial_data.csv', TASK_DIR / 'financial_data.csv',
    ]

    prolongation_path = next((p for p in prolongation_candidates if p.exists()), None)
    financial_path = next((p for p in financial_candidates if p.exists()), None)
    if prolongation_path and financial_path:
        return pd.read_csv(prolongation_path), pd.read_csv(financial_path), f'CSV: {prolongation_path.name}, {financial_path.name}'

    if REPORT_PATH.exists():
        return (
            pd.read_excel(REPORT_PATH, sheet_name='Исходные prolongations'),
            pd.read_excel(REPORT_PATH, sheet_name='Исходные financial'),
            f'Excel companion: {REPORT_PATH.name}'
        )

    return (
        read_embedded_csv(EMBEDDED_PROLONGATIONS_CSV_GZ_B64),
        read_embedded_csv(EMBEDDED_FINANCIAL_CSV_GZ_B64),
        'Embedded CSV snapshot inside notebook'
    )

In [ ]:
prolongations_raw, financial_raw, source_used = read_inputs()
print('Источник данных:', source_used)

# AM в prolongations.csv является первичным источником ответственного менеджера.
# Удаляем только полностью одинаковые строки id+month+AM, чтобы не задвоить один и тот же факт завершения.
duplicate_exact_count = int(prolongations_raw.duplicated(['id', 'month', 'AM']).sum())
prolongations = prolongations_raw.drop_duplicates(['id', 'month', 'AM']).copy()
prolongations['end_month'] = prolongations['month'].map(parse_period)

month_cols = [c for c in financial_raw.columns if c not in ['id', 'Причина дубля', 'Account']]
financial_amounts = financial_raw[['id']].copy()
for col in month_cols:
    financial_amounts[col] = financial_raw[col].map(parse_amount)

# В financial_data.csv один id может встречаться несколько раз из-за частей оплаты, допработ и смены ЮЛ.
# Для расчета пролонгации нужна полная отгрузка проекта за месяц, поэтому строки суммируются по id.
financial_by_id = financial_amounts.groupby('id', as_index=True)[month_cols].sum()
events = prolongations.merge(financial_by_id, left_on='id', right_index=True, how='left')
events[month_cols] = events[month_cols].fillna(0.0)

account_by_id = financial_raw.groupby('id')['Account'].agg(lambda s: sorted(set(s.dropna())))
account_check = prolongations.merge(account_by_id.rename('financial_accounts'), left_on='id', right_index=True, how='left')
id_month_conflicts = prolongations_raw.groupby(['id', 'month'])['AM'].nunique().reset_index(name='am_count')

quality = {
    'Строк prolongations.csv': len(prolongations_raw),
    'Строк после удаления точных дублей': len(prolongations),
    'Удалено точных дублей prolongations': duplicate_exact_count,
    'Конфликтов id+month с разными AM': int((id_month_conflicts['am_count'] > 1).sum()),
    'Строк financial_data.csv': len(financial_raw),
    'Уникальных id в financial_data.csv': int(financial_raw['id'].nunique()),
    'Дублирующихся строк id в financial_data.csv': int(financial_raw.duplicated('id').sum()),
    'id из prolongations без financial_data': len(set(prolongations['id']) - set(financial_raw['id'])),
    'id из financial_data без prolongations': len(set(financial_raw['id']) - set(prolongations['id'])),
    'Расхождений AM vs Account': int(account_check.apply(lambda r: r['AM'] not in (r['financial_accounts'] or []), axis=1).sum()),
}
quality_df = pd.DataFrame(quality.items(), columns=['Проверка', 'Значение'])
display(quality_df)

In [ ]:
def calculate_metrics(events: pd.DataFrame) -> pd.DataFrame:
    managers = sorted(events['AM'].dropna().unique().tolist())
    subjects = managers + ['Весь отдел']
    rows = []

    for subject in subjects:
        subject_events = events if subject == 'Весь отдел' else events[events['AM'].eq(subject)]
        for target_month in REPORT_MONTHS:
            prev_month = target_month - pd.DateOffset(months=1)
            prev2_month = target_month - pd.DateOffset(months=2)
            prev_label = month_label(prev_month)
            prev2_label = month_label(prev2_month)
            target_label = month_label(target_month)

            # Коэффициент 1: завершились в прошлом месяце, есть/нет отгрузка в текущем.
            cohort1 = subject_events[subject_events['end_month'].eq(prev_month)]
            k1_base = float(cohort1[prev_label].sum()) if prev_label in cohort1.columns else 0.0
            k1_extended = float(cohort1.loc[cohort1[target_label] > 0, target_label].sum()) if target_label in cohort1.columns else 0.0

            # Коэффициент 2: завершились два месяца назад, не продлились в первый месяц, но могут продлиться во второй.
            cohort2 = subject_events[subject_events['end_month'].eq(prev2_month)].copy()
            if prev_label in cohort2.columns:
                cohort2 = cohort2[cohort2[prev_label].le(0)]
            else:
                cohort2 = cohort2.iloc[0:0]
            k2_base = float(cohort2[prev2_label].sum()) if prev2_label in cohort2.columns else 0.0
            k2_extended = float(cohort2.loc[cohort2[target_label] > 0, target_label].sum()) if target_label in cohort2.columns else 0.0

            rows.append({
                'Менеджер': subject,
                'Месяц': target_label,
                'month_start': target_month,
                'К1 к пролонгации': k1_base,
                'К1 пролонгировано': k1_extended,
                'Коэффициент 1': coeff(k1_extended, k1_base),
                'К2 к пролонгации': k2_base,
                'К2 пролонгировано': k2_extended,
                'Коэффициент 2': coeff(k2_extended, k2_base),
            })
    return pd.DataFrame(rows)

monthly = calculate_metrics(events)
annual = monthly.groupby('Менеджер', as_index=False).agg({
    'К1 к пролонгации': 'sum',
    'К1 пролонгировано': 'sum',
    'К2 к пролонгации': 'sum',
    'К2 пролонгировано': 'sum',
})
annual['Коэффициент 1'] = annual.apply(lambda r: coeff(r['К1 пролонгировано'], r['К1 к пролонгации']), axis=1)
annual['Коэффициент 2'] = annual.apply(lambda r: coeff(r['К2 пролонгировано'], r['К2 к пролонгации']), axis=1)
annual = annual.sort_values('Менеджер')

def format_for_display(df):
    out = df.copy()
    for col in ['К1 к пролонгации', 'К1 пролонгировано', 'К2 к пролонгации', 'К2 пролонгировано']:
        out[col] = out[col].map(lambda x: f'{x:,.0f}'.replace(',', ' '))
    for col in ['Коэффициент 1', 'Коэффициент 2']:
        out[col] = out[col].map(lambda x: '' if pd.isna(x) else f'{x:.1%}')
    return out

print('Итог по отделу за 2023')
display(format_for_display(annual[annual['Менеджер'].eq('Весь отдел')]))
print('Помесячная динамика отдела')
display(format_for_display(monthly[monthly['Менеджер'].eq('Весь отдел')].drop(columns='month_start')))

## Экспорт

Оформленный управленческий Excel уже приложен отдельным файлом. Последняя ячейка нужна только если требуется выгрузить расчетные таблицы заново.


In [ ]:
# Необязательный экспорт расчетных таблиц.
# Основной оформленный отчет уже находится в _prolongation_report.xlsx.
EXPORT_RAW_TABLES = False
if EXPORT_RAW_TABLES:
    raw_export_path = TASK_DIR / '_prolongation_calculation_tables.xlsx'
    with pd.ExcelWriter(raw_export_path, engine='openpyxl') as writer:
        monthly.drop(columns='month_start').to_excel(writer, sheet_name='Детализация', index=False)
        annual.to_excel(writer, sheet_name='Менеджеры за год raw', index=False)
        quality_df.to_excel(writer, sheet_name='Проверки', index=False)
    print(f'Raw calculation tables written: {raw_export_path}')
else:
    print('Raw export skipped. Для экспорта поставьте EXPORT_RAW_TABLES = True.')